* Macro-averaged F1
* Balanced accuracy
* Per-class precision 
* Per-class recall
* Per-class F1-scores

# Libraries

In [1]:
import os
import sys
from pathlib import Path
sys.path.append(
    str(Path('..', 'utils_functionality', 'split_utils'))
)
sys.path.append(
    str(Path('..', 'utils_functionality', 'models'))
)

from modelling3_utils import MLPipeline
import joblib
import pandas as pd
from tqdm import tqdm
from IPython.display import clear_output

# Calculations

In [2]:
path_models = Path('../results/models_modelling3')

In [3]:
def get_additional_metrics(*, 
                           target, path_models, postfix='Stk_diam'):
    df_additional_metrics = pd.DataFrame()
    paths_models = [
        str(path_models / x).replace('\\', '/') for x\
            in os.listdir(path_models) if target in x and postfix in x and 'Logistic' not in x or '_kl' in x]
    for path_model in tqdm(paths_models):
        if 'Stump' in path_model:
            postfix = path_model.split(target)[-1]
        pipeline = joblib.load(path_model)
        model = pipeline.steps[3][1]
        ml_pipe = MLPipeline(
            target=target,
            estimator=model,
            features_to_drop = (),
            model_postfix=postfix,)
        ml_pipe.run(save_model_and_metrics=False)
        df_additional_metrics = pd.concat(
            (df_additional_metrics, ml_pipe.df_results))
        clear_output()
    return df_additional_metrics

In [4]:
target = 'splashing'
df_additional_metrics_spl = get_additional_metrics(
    target=target, path_models=path_models)

100%|██████████| 10/10 [00:15<00:00,  1.56s/it]


In [5]:
target = 'no_fragmentation'
df_additional_metrics_nof = get_additional_metrics(
    target=target, path_models=path_models)

100%|██████████| 10/10 [00:25<00:00,  2.51s/it]


In [6]:
pd.concat((
    df_additional_metrics_spl,
    df_additional_metrics_nof
)).to_excel('../results/metrics_modelling3_additional.xlsx', index=False)